In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql import functions as F, types as T

In [0]:
FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations?"
API_KEY = dbutils.secrets.get(scope="electricity", key="fred_api_key")

OBSERVATION_START = "2000-01-01"
OBSERVATION_END = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

CATALOG = "workspace"
SCHEMA = "us_electricity"
BRONZE_TABLE = "raw_fred_observations"

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE}"

FRED_SERIES = [
    "APU000072610",
    "CUSR0000SEHF01",
    "CPIAUCSL",
    "CPILFESL",
    "MHHNGSP"
]

def get_fred_observations(fred_series_id):
    response = requests.get(f"{FRED_BASE_URL}series_id={fred_series_id}&api_key={API_KEY}&file_type=json&observation_start={OBSERVATION_START}&observation_end={OBSERVATION_END}",timeout=30)
    response.raise_for_status()
    return response.json().get("observations", [])

def normalize_bronze_records(fred_series_id, observations):
    records = []

    for observation in observations:
        records.append({
            "series_id": fred_series_id,
            "observation_date_raw": observation.get("date"),
            "value_raw": observation.get("value"),
            "source_system": "FRED"
        })
    return records

if  __name__ == "__main__":

    all_records = []

    for series_id in FRED_SERIES:
        observations = get_fred_observations(series_id)
        all_records.extend(normalize_bronze_records(series_id, observations))
    
    bronze_schema = T.StructType([
        T.StructField("series_id", T.StringType(), False),
        T.StructField("observation_date_raw", T.StringType(), True),
        T.StructField("value_raw", T.StringType(), True),
        T.StructField("source_system", T.StringType(), False)
    ])

    bronze_df = spark.createDataFrame(all_records, bronze_schema)

    if bronze_df.isEmpty():
        raise ValueError(
            "Bronze DataFrame is empty. Refusing to overwrite the target table."
    )

    bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(TARGET_TABLE)


